#  Student Productivity Clustering — KMeans

**Dataset:** `ultimate_student_productivity_dataset_5000.csv`  
**Task:** Unsupervised Clustering — Segmentasi Mahasiswa berdasarkan Produktivitas  
**Model:** KMeans Clustering

---
###  Daftar Isi
1. Cara Melihat Tipe Data
2. Dataset Bisa Digunakan Untuk Apa
3. Model Yang Bisa Digunakan
4. Parameter Yang Bisa Diubah/Disetel
5. Evaluasi Yang Dipakai
6. Cara Mengetahui Evaluasi Bagus atau Tidak
7. Cara Mengoptimasi Model
8. Cara Menyimpan Model
9. Cara Menggunakan Model Hasil Training

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

print('Libraries loaded ')

---
## 1.  Cara Melihat Tipe Data

In [ ]:
df = pd.read_csv('../ultimate_student_productivity_dataset_5000.csv')
print(f'Shape: {df.shape}')
df.info()

In [ ]:
print(df.describe())

---
## 2.  Dataset Bisa Digunakan Untuk Apa

Clustering ≠ classification. Kita tidak punya label, kita ingin:
- **Menemukan segmen alami** dalam data mahasiswa
- Menjawab: "Profil mahasiswa seperti apa yang cenderung produktif?"
- Membantu dosen membuat intervensi yang tepat per kelompok

In [ ]:
drop_cols = ['student_id', 'productivity_score', 'exam_score']
df_proc = df.drop(columns=drop_cols).copy()
le = LabelEncoder()
for col in df_proc.select_dtypes(include='object').columns:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_proc)
print(f'Feature matrix shape: {X_scaled.shape}')

---
## 3.  Penjelasan KMeans

KMeans meminimasi **inertia** (sum of squared distances ke centroid):

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2$$

**Algoritma:**
1. Inisialisasi K centroid (KMeans++)
2. Assign setiap titik ke centroid terdekat
3. Update centroid = rata-rata titik dalam cluster
4. Ulangi hingga konvergen

**Kelebihan:** Cepat, mudah dipahami, scalable  
**Kekurangan:** Harus tentukan K manual, sensitif terhadap outlier, cluster harus berbentuk bulat

---
## 4.  Parameter Yang Bisa Diubah / Disetel

| Parameter | Default | Penjelasan |
|-----------|---------|------------|
| `n_clusters` | 8 | Jumlah cluster (K) |
| `init` | `'k-means++'` | Strategi inisialisasi centroid |
| `n_init` | 10 | Percobaan ulang (pilih inertia terkecil) |
| `max_iter` | 300 | Max iterasi per run |
| `algorithm` | `'lloyd'` | `'elkan'` lebih cepat untuk data padat |

---
## 5.  Evaluasi Yang Dipakai — Mencari K Optimal

In [ ]:
# Elbow Method + Silhouette Score
K_range = range(2, 11)
inertias, sil_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels, sample_size=2000, random_state=42))
    print(f'K={k}: Inertia={km.inertia_:.1f}, Silhouette={sil_scores[-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('Jumlah Cluster K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

best_k = list(K_range)[np.argmax(sil_scores)]
axes[1].plot(list(K_range), sil_scores, 's-', color='darkorange')
axes[1].axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
axes[1].set_xlabel('Jumlah Cluster K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score per K'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f'\nRekomendasi K optimal: {best_k}')

In [ ]:
# Training dengan K optimal
n_clusters = best_k
kmeans = KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=42)
df_proc['cluster'] = kmeans.fit_predict(X_scaled)
df['cluster'] = df_proc['cluster']
print(f'Cluster counts:\n{df["cluster"].value_counts().sort_index()}')

---
## 6.  Cara Mengetahui Evaluasi Bagus atau Tidak

| Metrik | Rentang | Interpretasi |
|--------|---------|-------------|
| **Silhouette Score** | -1 s/d 1 | > 0.5 = baik, > 0.7 = sangat baik |
| **Davies-Bouldin Index** | ≥ 0 | Semakin kecil semakin baik |
| **Inertia** | ≥ 0 | Semakin kecil semakin baik (trade-off dengan K) |

In [ ]:
labels = kmeans.labels_
sil = silhouette_score(X_scaled, labels, sample_size=2000, random_state=42)
dbi = davies_bouldin_score(X_scaled, labels)
print(f'Silhouette Score  : {sil:.4f}')
print(f'Davies-Bouldin    : {dbi:.4f}')
print(f'Inertia           : {kmeans.inertia_:.1f}')

---
## 7.  Cara Mengoptimasi Model — Interpretasi Cluster

In [ ]:
# Visualisasi dengan PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
palette = sns.color_palette('tab10', n_clusters)

plt.figure(figsize=(9, 6))
for c in range(n_clusters):
    mask = labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], s=10, alpha=0.5, label=f'Cluster {c}')
plt.title(f'KMeans Clusters (K={n_clusters}) — PCA 2D')
plt.legend(markerscale=3); plt.tight_layout(); plt.show()

In [ ]:
# Profil tiap cluster
numeric_cols = ['study_hours', 'sleep_hours', 'social_media_hours', 'mental_health_score',
                'focus_index', 'burnout_level', 'productivity_score']
profile = df.groupby('cluster')[numeric_cols].mean().round(2)
print(profile)

profile_norm = (profile - profile.min()) / (profile.max() - profile.min())
profile_norm.T.plot(kind='bar', figsize=(14, 5))
plt.title('Profil Cluster Ternormalisasi'); plt.xticks(rotation=30)
plt.legend(title='Cluster', bbox_to_anchor=(1,1)); plt.tight_layout(); plt.show()

---
## 8.  Cara Menyimpan Model

In [ ]:
os.makedirs('saved_models', exist_ok=True)
joblib.dump(kmeans, 'saved_models/kmeans_productivity.pkl')
joblib.dump(scaler, 'saved_models/scaler_kmeans.pkl')
joblib.dump(list(df_proc.drop(columns=['cluster']).columns), 'saved_models/feature_cols_kmeans.pkl')
print(f' KMeans (K={n_clusters}) dan scaler tersimpan!')

---
## 9.  Cara Menggunakan Model Hasil Training

In [ ]:
loaded_km  = joblib.load('saved_models/kmeans_productivity.pkl')
loaded_sc  = joblib.load('saved_models/scaler_kmeans.pkl')
feat_cols  = joblib.load('saved_models/feature_cols_kmeans.pkl')
print('Model dimuat ')

mahasiswa_baru = pd.DataFrame([{
    'age': 21, 'gender': 1, 'academic_level': 1, 'study_hours': 5, 'self_study_hours': 2,
    'online_classes_hours': 2, 'social_media_hours': 2, 'gaming_hours': 1, 'sleep_hours': 7,
    'screen_time_hours': 5, 'exercise_minutes': 30, 'caffeine_intake_mg': 150,
    'part_time_job': 0, 'upcoming_deadline': 1, 'internet_quality': 2,
    'mental_health_score': 7, 'focus_index': 6, 'burnout_level': 4
}])[feat_cols]

scaled_new = loaded_sc.transform(mahasiswa_baru)
cluster_id = loaded_km.predict(scaled_new)[0]
print(f'Mahasiswa baru masuk Cluster: {cluster_id}')
print(f'Profil cluster {cluster_id}:')
print(profile.loc[cluster_id])